# Análise Descritiva — Retail Store Sales Data

Este notebook reproduz os principais indicadores e cortes usados na análise descritiva (faturamento, lucro e margem) e prepara a base para as análises subsequentes.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path('..') / 'data' / 'raw' / 'Case 1 - Case Retail Store Sales Data.xlsx'
SHEET = 'Sales_retail_store'

df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
df.head()

In [ ]:
import unicodedata

def norm_ascii(s: str) -> str:
    s = '' if s is None else str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    return ' '.join(s.split()).lower()

rename = {}
for c in df.columns:
    k = norm_ascii(c)
    if k == 'regiao':
        rename[c] = 'Região'
    if k == 'preco unitario':
        rename[c] = 'Preço Unitário'
if rename:
    df = df.rename(columns=rename)

df['Data da Venda'] = pd.to_datetime(df['Data da Venda'])
df['Ano'] = df['Data da Venda'].dt.year
df['Mes'] = df['Data da Venda'].dt.to_period('M').astype(str)

for c in ['Faturamento', 'Lucro', 'Desconto', 'Custo de Envio']:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

df.shape

In [ ]:
fat_total = df['Faturamento'].sum()
luc_total = df['Lucro'].sum()
marg_total = luc_total / fat_total if fat_total else np.nan

resumo = pd.DataFrame(
    {
        'Faturamento (R$)': [fat_total],
        'Lucro (R$)': [luc_total],
        'Margem (%)': [marg_total * 100],
        'Pedidos': [df['Order ID'].nunique()],
        'Linhas': [len(df)],
    }
)
resumo

In [ ]:
por_ano = (
    df.groupby('Ano', as_index=False)
      .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'), custo_envio=('Custo de Envio','sum'))
)
por_ano['margem'] = np.where(por_ano['faturamento'] != 0, por_ano['lucro'] / por_ano['faturamento'], np.nan)
por_ano['custo_envio_pct'] = np.where(por_ano['faturamento'] != 0, por_ano['custo_envio'] / por_ano['faturamento'], np.nan)
por_ano.sort_values('Ano')

In [ ]:
por_mes = (
    df.groupby('Mes', as_index=False)
      .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
por_mes['margem'] = np.where(por_mes['faturamento'] != 0, por_mes['lucro']/por_mes['faturamento'], np.nan)
por_mes.tail(12)

In [ ]:
por_cat = (
    df.groupby('Categoria do Produto', as_index=False)
      .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
por_cat['margem'] = np.where(por_cat['faturamento'] != 0, por_cat['lucro']/por_cat['faturamento'], np.nan)
por_cat.sort_values('lucro')

In [ ]:
por_sub = (
    df.groupby(['Categoria do Produto','Sub-Categoria do Produto'], as_index=False)
      .agg(faturamento=('Faturamento','sum'), lucro=('Lucro','sum'))
)
por_sub['margem'] = np.where(por_sub['faturamento'] != 0, por_sub['lucro']/por_sub['faturamento'], np.nan)
por_sub.sort_values('lucro').head(10)

## Pedidos com prejuízo
A incidência de pedidos com lucro negativo é um indicador de vazamento de margem e é usado como peça central no diagnóstico e nas prescrições (guardrails).

In [ ]:
order = df.groupby('Order ID', as_index=False).agg(fat=('Faturamento','sum'), lucro=('Lucro','sum'))
neg = order[order['lucro'] < 0]
pd.DataFrame({
    'Pedidos totais': [len(order)],
    'Pedidos com prejuízo': [len(neg)],
    '% pedidos com prejuízo': [len(neg)/max(1,len(order))],
    '% faturamento em pedidos com prejuízo': [neg['fat'].sum() / (order['fat'].sum() or 1)],
})